# Прогноз арендной платы за квартиру в Москве — полный ML-пайплайн (Google Colab)

Один ноутбук, воспроизводящий весь пайплайн репозитория [`econ-housing-price-ml`](https://github.com/jamik-ai/econ-housing-price-ml) — от скачивания данных до итогового отчёта с ответами на гипотезы H1–H5. Полная постановка задачи, источники данных и методология — в [`ТЗ.md`](https://github.com/jamik-ai/econ-housing-price-ml/blob/main/ТЗ.md) репозитория; этот ноутбук пересобирает тот же пайплайн (`src/02_clean_data.py` … `07_visualize.py`) последовательными ячейками для запуска в Colab без клонирования репозитория.

**Порядок:** установка зависимостей → скачивание данных (Kaggle API) → очистка → feature engineering → обучение моделей → оценка → интерпретация (SHAP) → визуализация → итоговый отчёт.


Устанавливаем зависимости, которых нет в Colab по умолчанию (`pandas`/`numpy`/`scikit-learn`/`matplotlib`/`seaborn` там уже есть).


In [ ]:
!pip install -q kaggle lightgbm shap


Импорты и пути проекта — структура папок повторяет репозиторий (`data/raw`, `data/processed`, `output/figures`, `output/tables`).


In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
MODELS_DIR = PROCESSED_DIR / "models"
TABLES_DIR = Path("output/tables")
FIG_DIR = Path("output/figures")
for d in (RAW_DIR, PROCESSED_DIR, MODELS_DIR, TABLES_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)


## Данные

Источник — `beldmian/moscow-rent-cian-with-images` (Kaggle, лицензия CC0-1.0): прямой скрапинг объявлений об аренде квартир в Москве с cian.ru. Полная история выбора источника (включая отклонённый синтетический датасет) и аудит схемы — в [`output/tables/data_audit.md`](https://github.com/jamik-ai/econ-housing-price-ml/blob/main/output/tables/data_audit.md) репозитория.

Нужен бесплатный личный токен Kaggle: https://www.kaggle.com/settings/api


In [ ]:
import os
from getpass import getpass

if not os.environ.get("KAGGLE_API_TOKEN"):
    os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token: ")

!kaggle datasets download beldmian/moscow-rent-cian-with-images -f moscow_rent_all.csv -p data/raw

import zipfile

zip_path = RAW_DIR / "moscow_rent_all.csv.zip"
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(RAW_DIR)


Быстрая проверка структуры скачанных данных (объём, доля пропусков) — полный аудит с историей выбора источника см. в `output/tables/data_audit.md` репозитория.


In [ ]:
df_raw = pd.read_csv(RAW_DIR / "moscow_rent_all.csv", sep=";", low_memory=False)
print(f"Строк: {len(df_raw)}, столбцов: {len(df_raw.columns)}")
df_raw.isna().mean().sort_values(ascending=False).head(10)


## Этап 2 — очистка данных

- Удаляются почти пустые столбцы (`heating_type`, `ceiling_height_m`, `parking_type`, `passenger_lifts_count`, >99% пропусков) и неиспользуемые текстовые/идентификационные столбцы.
- `rooms`: пропуски → 0 (студия), с явным флагом `rooms_was_missing`.
- Строки без станции метро (0.3%) — удаляются, а не имитируются.
- `living_area_sqm`/`kitchen_area_sqm`/`build_year`: пропуски → медиана по группе `rooms`, с флагами `*_missing`.
- `house_material`/`repair_type`: пропуски → явная категория `unknown`.
- `author_type` → бинарный флаг `is_agent` (заполнен только для агентов).
- Выбросы по `price_rub`/`area_sqm` — отсечение по 1–99 перцентилю.
- Целевая переменная — `log_price_rub = log1p(price_rub)` (распределение цены сильно скошено).


In [ ]:
NEAR_EMPTY_COLS = ["heating_type", "ceiling_height_m", "parking_type", "passenger_lifts_count"]
UNUSED_COLS = [
    "url", "title", "description", "detail_features", "image_urls", "image_paths",
    "images_count", "deal_type", "price_formatted", "repair", "address",
    "residential_complex", "loggias_count", "balconies_count", "combined_wcs_count",
    "separate_wcs_count", "pets_allowed", "children_allowed", "utilities_included",
    "details_fetched",
]

df_clean = df_raw.drop(columns=NEAR_EMPTY_COLS + UNUSED_COLS, errors="ignore")
n_raw = len(df_raw)
cleaning_report = [f"# Отчёт об очистке данных\n", f"Исходных строк: {n_raw}\n"]

df_clean["rooms_was_missing"] = df_clean["rooms"].isnull().astype(int)
df_clean["rooms"] = df_clean["rooms"].fillna(0)

n_before = len(df_clean)
df_clean = df_clean.dropna(subset=["metro", "metro_time_min"])
cleaning_report.append(f"Удалены строки без станции метро: {n_before - len(df_clean)}\n")

for col in ["living_area_sqm", "kitchen_area_sqm", "build_year"]:
    df_clean[f"{col}_missing"] = df_clean[col].isnull().astype(int)
    medians = df_clean.groupby("rooms")[col].transform("median")
    df_clean[col] = df_clean[col].fillna(medians).fillna(df_clean[col].median())

for col in ["house_material", "repair_type"]:
    df_clean[col] = df_clean[col].fillna("unknown")

df_clean["is_agent"] = (df_clean["author_type"] == "agent").astype(int)
df_clean = df_clean.drop(columns=["author_type", "author"], errors="ignore")

for col in ["deposit_rub", "prepay_months"]:
    df_clean[f"{col}_missing"] = df_clean[col].isnull().astype(int)
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

n_before = len(df_clean)
for col in ["price_rub", "area_sqm"]:
    lo, hi = df_clean[col].quantile([0.01, 0.99])
    df_clean = df_clean[(df_clean[col] >= lo) & (df_clean[col] <= hi)]
cleaning_report.append(
    f"Отсечены выбросы по 1-99 перцентилю price_rub/area_sqm: удалено {n_before - len(df_clean)} строк "
    f"({(n_before - len(df_clean)) / n_before:.1%})\n"
)

df_clean["log_price_rub"] = np.log1p(df_clean["price_rub"])
cleaning_report.append(f"Итоговых строк: {len(df_clean)} (было {n_raw})\n")

(TABLES_DIR / "cleaning_report.md").write_text("\n".join(cleaning_report), encoding="utf-8")
df_clean.to_csv(PROCESSED_DIR / "cleaned.csv", index=False)
print(f"{len(df_clean)} строк x {len(df_clean.columns)} столбцов -> {PROCESSED_DIR / 'cleaned.csv'}")


### Проверка на утечку целевой переменной

`deposit_rub` выглядит как обычный признак объекта, но депозит в объявлениях об аренде почти всегда равен размеру арендной платы — это нужно явно проверить перед тем, как использовать его как признак модели.


In [ ]:
corr_deposit = df_clean["deposit_rub"].corr(df_clean["price_rub"])
ratio_median = (df_clean["deposit_rub"] / df_clean["price_rub"]).median()
print(f"corr(deposit_rub, price_rub) = {corr_deposit:.2f}")
print(f"медиана(deposit_rub / price_rub) = {ratio_median:.2f}")


corr ≈0.9 и медианное отношение ≈1.0 → это прямая утечка целевой переменной, а не структурный признак объекта. `deposit_rub` исключён из набора признаков модели на следующем шаге.


## Этап 3 — feature engineering

- `price_per_sqm = price_rub / area_sqm` и `price_segment` (квартили цены за м²: econom/comfort/business/premium) — только для пост-хок сравнения по сегментам (H4), не используются как признак модели.
- Редкие категории `district`/`metro` (<20 объявлений) группируются в `Other` перед target encoding.
- `deposit_rub` исключён из признаков (см. проверку утечки выше).


In [ ]:
RARE_THRESHOLD = 20

def group_rare_categories(series, threshold):
    counts = series.value_counts()
    rare = counts[counts < threshold].index
    return series.where(~series.isin(rare), other="Other")

df_features = df_clean.copy()
df_features["price_per_sqm"] = df_features["price_rub"] / df_features["area_sqm"]
df_features["price_segment"] = pd.qcut(
    df_features["price_per_sqm"], q=4, labels=["econom", "comfort", "business", "premium"]
)
df_features["district_grouped"] = group_rare_categories(df_features["district"], RARE_THRESHOLD)
df_features["metro_grouped"] = group_rare_categories(df_features["metro"], RARE_THRESHOLD)

TARGET_COL = "log_price_rub"
NUMERIC_FEATURES = [
    "rooms", "area_sqm", "living_area_sqm", "kitchen_area_sqm", "floor", "floors_total",
    "metro_time_min", "build_year", "prepay_months", "rooms_was_missing",
    "living_area_sqm_missing", "kitchen_area_sqm_missing", "build_year_missing",
    "prepay_months_missing", "is_agent",
]
CATEGORICAL_FEATURES = ["repair_type", "house_material", "district_grouped", "metro_grouped"]

df_features.to_csv(PROCESSED_DIR / "features.csv", index=False)
print(
    f"district: {df_clean['district'].nunique()} -> {df_features['district_grouped'].nunique()} категорий; "
    f"metro: {df_clean['metro'].nunique()} -> {df_features['metro_grouped'].nunique()} категорий"
)


## Этап 4 — обучение моделей

Baseline (Ridge) + RandomForest + LightGBM, с подбором гиперпараметров через `GridSearchCV` и кросс-валидацией. Высококардинальные `district_grouped`/`metro_grouped` кодируются через target encoding по статистике train, остальные категории — через one-hot.

⚠️ На CPU Colab обучение может занять 5–15 минут (GridSearchCV для RandomForest и LightGBM).

LightGBM обучается с `deterministic=True, force_row_wise=True` — без этого многопоточное построение гистограмм даёт чуть разные деревья между перезапусками даже при фиксированном `random_state` (на это наткнулись при тестировании: вывод по H4, где разница между сегментами небольшая, между запусками менялся).


In [ ]:
TARGET_ENCODE_SMOOTHING = 20

def target_encode(train_col, train_target, other_col, smoothing):
    global_mean = train_target.mean()
    stats = train_target.groupby(train_col).agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
    mapping = smoothed.to_dict()
    train_encoded = train_col.map(mapping).fillna(global_mean)
    other_encoded = other_col.map(mapping).fillna(global_mean)
    return train_encoded, other_encoded

def build_design_matrix(df_train, df_test, numeric_features, categorical_features, target_col):
    X_train = df_train[numeric_features].copy()
    X_test = df_test[numeric_features].copy()
    for col in ["district_grouped", "metro_grouped"]:
        enc_train, enc_test = target_encode(df_train[col], df_train[target_col], df_test[col], TARGET_ENCODE_SMOOTHING)
        X_train[f"{col}_te"] = enc_train.values
        X_test[f"{col}_te"] = enc_test.values
    low_card_cols = [c for c in categorical_features if c not in ("district_grouped", "metro_grouped")]
    train_dummies = pd.get_dummies(df_train[low_card_cols], prefix=low_card_cols)
    test_dummies = pd.get_dummies(df_test[low_card_cols], prefix=low_card_cols).reindex(columns=train_dummies.columns, fill_value=0)
    X_train = pd.concat([X_train.reset_index(drop=True), train_dummies.reset_index(drop=True)], axis=1)
    X_test = pd.concat([X_test.reset_index(drop=True), test_dummies.reset_index(drop=True)], axis=1)
    return X_train, X_test

df_train, df_test = train_test_split(
    df_features, test_size=0.2, random_state=RANDOM_STATE, stratify=df_features["price_segment"]
)
print(f"train: {len(df_train)} test: {len(df_test)}")

X_train, X_test = build_design_matrix(df_train, df_test, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_COL)
y_train, y_test = df_train[TARGET_COL].values, df_test[TARGET_COL].values
feature_names = X_train.columns.tolist()

models = {}

ridge = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())])
ridge_grid = GridSearchCV(ridge, {"ridge__alpha": [0.1, 1.0, 10.0, 50.0]}, cv=5, scoring="neg_root_mean_squared_error")
ridge_grid.fit(X_train, y_train)
models["ridge"] = ridge_grid.best_estimator_
print("Ridge best params:", ridge_grid.best_params_)

rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
rf_grid = GridSearchCV(
    rf, {"n_estimators": [200], "max_depth": [12, 20, None], "min_samples_leaf": [1, 5]},
    cv=3, scoring="neg_root_mean_squared_error",
)
rf_grid.fit(X_train, y_train)
models["random_forest"] = rf_grid.best_estimator_
print("RandomForest best params:", rf_grid.best_params_)

lgbm = LGBMRegressor(random_state=RANDOM_STATE, verbose=-1, deterministic=True, force_row_wise=True)
lgbm_grid = GridSearchCV(
    lgbm, {"n_estimators": [300, 600], "num_leaves": [15, 31], "learning_rate": [0.05, 0.1]},
    cv=3, scoring="neg_root_mean_squared_error",
)
lgbm_grid.fit(X_train, y_train)
models["lightgbm"] = lgbm_grid.best_estimator_
print("LightGBM best params:", lgbm_grid.best_params_)

for name, model in models.items():
    joblib.dump(model, MODELS_DIR / f"{name}.joblib")

predictions = df_test[["offer_id", "price_rub", "price_segment"]].copy().reset_index(drop=True)
predictions["log_price_rub_actual"] = y_test
for name, model in models.items():
    pred_log = model.predict(X_test)
    predictions[f"{name}_pred_log"] = pred_log
    predictions[f"{name}_pred_rub"] = np.expm1(pred_log)
predictions.to_csv(PROCESSED_DIR / "predictions.csv", index=False)

X_train.assign(**{TARGET_COL: y_train}).to_csv(PROCESSED_DIR / "model_input_train.csv", index=False)
X_test.assign(**{TARGET_COL: y_test}).to_csv(PROCESSED_DIR / "model_input_test.csv", index=False)
(TABLES_DIR / "feature_names.json").write_text(json.dumps(feature_names, indent=2, ensure_ascii=False), encoding="utf-8")
print("Готово: модели и предсказания сохранены")


## Этап 5 — оценка качества моделей

Метрики на test (RMSE, MAE, RMSLE, R²) + проверка гетероскедастичности остатков лучшей модели.


In [ ]:
MODEL_NAMES = ["ridge", "random_forest", "lightgbm"]
y_true_rub = predictions["price_rub"].values
y_true_log = predictions["log_price_rub_actual"].values

rows = []
for name in MODEL_NAMES:
    pred_rub = predictions[f"{name}_pred_rub"].values
    pred_log = predictions[f"{name}_pred_log"].values
    rows.append({
        "model": name,
        "RMSE_rub": np.sqrt(mean_squared_error(y_true_rub, pred_rub)),
        "MAE_rub": mean_absolute_error(y_true_rub, pred_rub),
        "RMSLE": np.sqrt(mean_squared_error(y_true_log, pred_log)),
        "R2_rub": r2_score(y_true_rub, pred_rub),
        "R2_log": r2_score(y_true_log, pred_log),
    })
metrics = pd.DataFrame(rows).sort_values("RMSLE").reset_index(drop=True)
metrics.to_csv(TABLES_DIR / "model_metrics.csv", index=False)
metrics


Фиксируем лучшую модель по RMSLE и считаем диагностику остатков (проверка гетероскедастичности — растёт ли ошибка с ценой).


In [ ]:
best_model_name = metrics.iloc[0]["model"]
pred_rub_best = predictions[f"{best_model_name}_pred_rub"].values
residuals_rub = y_true_rub - pred_rub_best
abs_resid_corr = np.corrcoef(np.abs(residuals_rub), y_true_rub)[0, 1]

residuals = predictions[["offer_id", "price_segment", "price_rub"]].copy()
residuals["pred_rub"] = pred_rub_best
residuals["residual_rub"] = residuals_rub
residuals.to_csv(PROCESSED_DIR / "best_model_residuals.csv", index=False)
(TABLES_DIR / "best_model.txt").write_text(best_model_name, encoding="utf-8")

print(f"Лучшая модель: {best_model_name}")
print(f"Корреляция |остаток| с фактической ценой: {abs_resid_corr:.3f}")


## Этап 6 — интерпретация (SHAP, permutation importance, partial dependence, H4)


In [ ]:
model = models[best_model_name]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names).sort_values(ascending=False)
mean_abs_shap.to_frame("mean_abs_shap").to_csv(TABLES_DIR / "shap_importance.csv")
np.save(PROCESSED_DIR / "shap_values_test.npy", shap_values)
mean_abs_shap.head(10)


Permutation importance (модель-агностичная проверка SHAP), partial dependence по площади и времени до метро, и сравнение важности `metro_time_min` между ценовыми сегментами (H4).


In [ ]:
perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_importance = pd.Series(perm.importances_mean, index=feature_names).sort_values(ascending=False)
perm_importance.to_frame("perm_importance_mean").to_csv(TABLES_DIR / "permutation_importance.csv")

overlap = len(set(mean_abs_shap.head(5).index) & set(perm_importance.head(5).index))
print(f"SHAP vs permutation importance: {overlap}/5 совпадений в топ-5")

pd_results = {}
for feat in ["area_sqm", "metro_time_min"]:
    pdp = partial_dependence(model, X_test, [feat], kind="average", grid_resolution=30)
    pd_results[feat] = {"grid": pdp["grid_values"][0].tolist(), "average": pdp["average"][0].tolist()}
(PROCESSED_DIR / "partial_dependence.json").write_text(json.dumps(pd_results), encoding="utf-8")

segments = predictions["price_segment"].reset_index(drop=True)
feat_idx = feature_names.index("metro_time_min")
econom_shap = np.abs(shap_values[(segments == "econom").values, feat_idx]).mean()
premium_shap = np.abs(shap_values[(segments == "premium").values, feat_idx]).mean()
h4_confirmed = econom_shap > premium_shap
print(f"H4: mean|SHAP(metro_time_min)| econom={econom_shap:.4f}, premium={premium_shap:.4f} -> "
      f"{'подтверждается' if h4_confirmed else 'не подтверждается'}")


## Этап 7 — визуализация


In [ ]:
def savefig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=120)
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df_features["price_rub"], bins=60, ax=axes[0])
axes[0].set_title("Распределение арендной платы, ₽/мес.")
sns.histplot(np.log1p(df_features["price_rub"]), bins=60, ax=axes[1])
axes[1].set_title("log1p(price_rub)")
savefig("01_price_distribution.png")

plt.figure(figsize=(7, 4))
sns.histplot(df_features["price_per_sqm"].clip(upper=df_features["price_per_sqm"].quantile(0.99)), bins=60)
plt.title("Распределение цены за м² (обрезано по 99-му перцентилю)")
savefig("02_price_per_sqm_distribution.png")

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_features.sample(min(5000, len(df_features)), random_state=RANDOM_STATE), x="area_sqm", y="price_rub", alpha=0.3, s=15)
plt.title("Арендная плата vs площадь")
savefig("03_price_vs_area_scatter.png")

plt.figure(figsize=(7, 4))
order = ["no", "cosmetic", "euro", "design", "unknown"]
sns.boxplot(data=df_features, x="repair_type", y="price_per_sqm", order=order, showfliers=False)
plt.title("Цена за м² по типу ремонта (H2)")
savefig("04_price_per_sqm_by_repair.png")

plt.figure(figsize=(8, 4))
sns.boxplot(data=df_features, x="house_material", y="price_per_sqm", showfliers=False)
plt.title("Цена за м² по материалу дома")
plt.xticks(rotation=30, ha="right")
savefig("05_price_per_sqm_by_material.png")

top_districts = df_features.groupby("district")["price_per_sqm"].mean().sort_values(ascending=False).head(15)
plt.figure(figsize=(8, 5))
sns.barplot(x=top_districts.values, y=top_districts.index, color="steelblue")
plt.title("Топ-15 районов по средней цене за м²")
plt.xlabel("₽/м²")
savefig("06_top_districts_price_per_sqm.png")

top_shap = mean_abs_shap.head(15)
plt.figure(figsize=(8, 6))
sns.barplot(x=top_shap.values, y=top_shap.index, color="darkorange")
plt.title(f"Важность признаков (mean |SHAP|), модель {best_model_name}")
savefig("07_shap_feature_importance.png")

plt.figure(figsize=(6, 6))
plt.scatter(predictions["price_rub"], predictions[f"{best_model_name}_pred_rub"], alpha=0.3, s=10)
lims = [0, predictions["price_rub"].quantile(0.99)]
plt.plot(lims, lims, color="red", linestyle="--")
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel("Фактическая цена, ₽/мес.")
plt.ylabel("Прогноз, ₽/мес.")
plt.title(f"Predicted vs Actual ({best_model_name})")
savefig("08_predicted_vs_actual.png")

plt.figure(figsize=(7, 5))
plt.scatter(residuals["pred_rub"], residuals["residual_rub"], alpha=0.3, s=10)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Прогноз, ₽/мес.")
plt.ylabel("Остаток, ₽/мес.")
plt.title("Остатки модели vs прогноз")
savefig("09_residuals_vs_predicted.png")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(pd_results["area_sqm"]["grid"], pd_results["area_sqm"]["average"])
axes[0].set_title("Partial dependence: area_sqm")
axes[0].set_xlabel("м²")
axes[0].set_ylabel("log_price_rub (предсказание)")
axes[1].plot(pd_results["metro_time_min"]["grid"], pd_results["metro_time_min"]["average"])
axes[1].set_title("Partial dependence: metro_time_min")
axes[1].set_xlabel("мин. до метро")
savefig("10_partial_dependence.png")


## Итоговый отчёт — ответы на гипотезы


In [ ]:
from IPython.display import Markdown, display

corr_area = df_features["area_sqm"].corr(df_features["price_rub"])
corr_rooms = df_features["rooms"].corr(df_features["price_rub"])
area_rank = list(mean_abs_shap.index).index("area_sqm") + 1
rooms_rank = list(mean_abs_shap.index).index("rooms") + 1
district_rank = list(mean_abs_shap.index).index("district_grouped_te") + 1
metro_rank = list(mean_abs_shap.index).index("metro_grouped_te") + 1

repair_means = df_features.groupby("repair_type")["price_per_sqm"].mean().reindex(["no", "cosmetic", "euro", "design"])
pd_metro_grid = pd_results["metro_time_min"]["grid"]
pd_metro_avg = pd_results["metro_time_min"]["average"]
metro_trend_decreasing = np.corrcoef(pd_metro_grid, pd_metro_avg)[0, 1] < 0

best_row = metrics.iloc[0]
ridge_row = metrics[metrics["model"] == "ridge"].iloc[0]

md_text = f"""
**H1 — площадь и число комнат — главные предикторы.**
`area_sqm` — SHAP-ранг {area_rank} (corr с ценой {corr_area:.2f}), `rooms` — SHAP-ранг {rooms_rank} (corr {corr_rooms:.2f}).
Район (ранг {district_rank}) и станция метро (ранг {metro_rank}) важнее числа комнат — площадь как главный фактор подтверждена, число комнат — нет.

**H2 — время до метро снижает цену, ремонт повышает.**
Partial dependence по `metro_time_min` {'убывает' if metro_trend_decreasing else 'растёт'} с увеличением времени до метро (с {pd_metro_avg[0]:.2f} до {pd_metro_avg[-1]:.2f} в лог-шкале).
Средняя цена за м² по ремонту: {' → '.join(f'{idx} {val:,.0f} ₽'.replace(',', ' ') for idx, val in repair_means.items())}.

**H3 — градиентный бустинг превосходит линейную регрессию.**
{best_row['model']}: R²={best_row['R2_rub']:.3f}, RMSLE={best_row['RMSLE']:.3f} vs Ridge: R²={ridge_row['R2_rub']:.3f}, RMSLE={ridge_row['RMSLE']:.3f}.

**H4 — близость к метро важнее в дешёвом сегменте.**
mean|SHAP(metro_time_min)|: econom={econom_shap:.4f}, premium={premium_shap:.4f} → гипотеза {'подтверждается' if h4_confirmed else 'не подтверждается'}.

**Гетероскедастичность остатков:** корреляция |остаток| с фактической ценой = {abs_resid_corr:.3f}.
"""
display(Markdown(md_text))


**H5 — связь с ключевой ставкой ЦБ.** Не проверяется: датасет почти весь сосредоточен в мае–июне 2026 (см. [`output/tables/data_audit.md`](https://github.com/jamik-ai/econ-housing-price-ml/blob/main/output/tables/data_audit.md)) — недостаточный и неоднородный временной охват для содержательной проверки связи с динамикой ключевой ставки. Гипотеза явно снята по результатам аудита данных, а не имитирована на недостаточных основаниях.

## Ограничения

- Цены — заявленные (asking rent), не фактически согласованные ставки.
- `deposit_rub` исключён из признаков как утечка целевой переменной (см. проверку выше) — без него R² честный (~0.88), а не искусственно завышенный (~0.97 с утечкой).
- Множество потенциально интересных признаков почти полностью отсутствуют (`heating_type`, `parking_type`, `ceiling_height_m`, `passenger_lifts_count`) — исключены, а не имитированы.
- `district`/`metro` — высококардинальные категории, закодированы target encoding по статистике train — возможен небольшой остаточный оптимизм оценки.
- Датасет — почти одномоментный срез рынка, не полноценный временной ряд.

Полная постановка задачи и методология — [`ТЗ.md`](https://github.com/jamik-ai/econ-housing-price-ml/blob/main/ТЗ.md) в репозитории.
